In [6]:
%pip install -q xgboost fairlearn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from itertools import combinations

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference
)

print("Imports completed.")

Imports completed.


In [8]:
# ==========================================
# CONFIGURATION
# ==========================================

DATA_PATH = "preprocessed-hmda.csv"

RANDOM_STATE = 42

TEST_SIZE = 0.20

# 5-Fold Cross Validation
CV_FOLDS = 5

# Number of hyperparameter combinations
N_ITER = 10

# Parallelism
XGB_N_JOBS = -1
SEARCH_N_JOBS = 1

# Output directory
OUTPUT_DIR = "xgb_100k_baseline_fairness"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Configuration ready.")
print("CV folds:", CV_FOLDS)
print("Random search iterations:", N_ITER)

Configuration ready.
CV folds: 5
Random search iterations: 10


In [9]:
# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nTarget distribution (%):")
print(
    df["target"]
    .value_counts(normalize=True)
)

Dataset shape: (100000, 15)

Columns:
['derived_ethnicity', 'derived_race', 'derived_sex', 'applicant_age', 'derived_dwelling_category', 'loan_purpose', 'conforming_loan_limit', 'loan_amount', 'loan_to_value_ratio', 'loan_term', 'property_value', 'occupancy_type', 'income', 'debt_to_income_ratio', 'target']

Target distribution:
target
1    76745
0    23255
Name: count, dtype: int64

Target distribution (%):
target
1    0.76745
0    0.23255
Name: proportion, dtype: float64


In [10]:
# ==========================================
# VERIFY MISSING VALUES
# ==========================================

missing_values = df.isna().sum()

print("Missing values:")
print(
    missing_values[
        missing_values > 0
    ]
)

print(
    "\nTotal missing values:",
    missing_values.sum()
)

Missing values:
Series([], dtype: int64)

Total missing values: 0


In [11]:
# ==========================================
# PROTECTED ATTRIBUTES
# ==========================================

protected_attributes = [
    "derived_ethnicity",
    "derived_race",
    "derived_sex",
    "applicant_age"
]

print("Protected attributes:")
print(protected_attributes)

Protected attributes:
['derived_ethnicity', 'derived_race', 'derived_sex', 'applicant_age']


In [12]:
# ==========================================
# MODEL FEATURES
# ==========================================

model_features = [
    "derived_dwelling_category",
    "loan_purpose",
    "conforming_loan_limit",
    "loan_amount",
    "loan_to_value_ratio",
    "loan_term",
    "property_value",
    "occupancy_type",
    "income",
    "debt_to_income_ratio"
]

print("Model features:")
for feature in model_features:
    print("-", feature)

Model features:
- derived_dwelling_category
- loan_purpose
- conforming_loan_limit
- loan_amount
- loan_to_value_ratio
- loan_term
- property_value
- occupancy_type
- income
- debt_to_income_ratio


In [13]:
# ==========================================
# VERIFY PROTECTED FEATURES ARE EXCLUDED
# ==========================================

overlap = set(
    model_features
).intersection(
    protected_attributes
)

print(
    "\nProtected attributes in X:",
    overlap
)

assert len(overlap) == 0

print(
    "✓ Protected attributes successfully excluded."
)


Protected attributes in X: set()
✓ Protected attributes successfully excluded.


In [14]:
# ==========================================
# X / y / PROTECTED DATA
# ==========================================

X = df[
    model_features
].copy()

y = df[
    "target"
].copy()

protected_data = df[
    protected_attributes
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(
    "Protected data shape:",
    protected_data.shape
)

X shape: (100000, 10)
y shape: (100000,)
Protected data shape: (100000, 4)


In [15]:
# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,

    test_size=TEST_SIZE,

    random_state=RANDOM_STATE,

    stratify=y
)

# Keep protected attributes corresponding
# to the exact test rows
protected_train = protected_data.loc[
    X_train.index
].copy()

protected_test = protected_data.loc[
    X_test.index
].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print(
    "protected_train:",
    protected_train.shape
)

print(
    "protected_test:",
    protected_test.shape
)

X_train: (80000, 10)
X_test : (20000, 10)
y_train: (80000,)
y_test : (20000,)
protected_train: (80000, 4)
protected_test: (20000, 4)


In [16]:
# ==========================================
# FEATURE TYPES
# ==========================================

categorical_features = [
    "derived_dwelling_category",
    "loan_purpose",
    "conforming_loan_limit",
    "occupancy_type"
]

numerical_features = [
    "loan_amount",
    "loan_to_value_ratio",
    "loan_term",
    "property_value",
    "income",
    "debt_to_income_ratio"
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['derived_dwelling_category', 'loan_purpose', 'conforming_loan_limit', 'occupancy_type']

Numerical features:
['loan_amount', 'loan_to_value_ratio', 'loan_term', 'property_value', 'income', 'debt_to_income_ratio']


In [17]:
# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

print("Preprocessor created.")

Preprocessor created.


In [18]:
# ==========================================
# XGBOOST MODEL
# ==========================================

xgb_model = XGBClassifier(
    objective="binary:logistic",

    eval_metric="logloss",

    tree_method="hist",

    random_state=RANDOM_STATE,

    n_jobs=XGB_N_JOBS
)

print("XGBoost model created.")

XGBoost model created.


In [19]:
# ==========================================
# XGBOOST PIPELINE
# ==========================================

xgb_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),

    (
        "model",
        xgb_model
    )
])

print("Pipeline created.")

Pipeline created.


In [20]:
# ==========================================
# HYPERPARAMETER SEARCH SPACE
# ==========================================

param_distributions = {

    "model__n_estimators": [
        100,
        200,
        300,
        500,
        700
    ],

    "model__max_depth": [
        3,
        4,
        5,
        6,
        8,
        10
    ],

    "model__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.08,
        0.10,
        0.15
    ],

    "model__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "model__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "model__min_child_weight": [
        1,
        3,
        5,
        10
    ],

    "model__gamma": [
        0,
        0.1,
        0.3,
        0.5
    ],

    "model__reg_alpha": [
        0,
        0.01,
        0.1,
        1
    ],

    "model__reg_lambda": [
        1,
        2,
        5,
        10
    ]
}

print(
    "Number of possible combinations:",
    np.prod([
        len(v)
        for v in param_distributions.values()
    ])
)

Number of possible combinations: 737280


In [21]:
# ==========================================
# 5-FOLD STRATIFIED CROSS VALIDATION
# ==========================================

cv_strategy = StratifiedKFold(
    n_splits=5,

    shuffle=True,

    random_state=RANDOM_STATE
)

print("5-Fold Stratified CV ready.")

5-Fold Stratified CV ready.


In [22]:
# ==========================================
# RANDOMIZED SEARCH
# ==========================================

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,

    param_distributions=param_distributions,

    n_iter=N_ITER,

    scoring="accuracy",

    cv=cv_strategy,

    random_state=RANDOM_STATE,

    n_jobs=SEARCH_N_JOBS,

    verbose=2,

    return_train_score=False,

    refit=True
)

print("RandomizedSearchCV created.")

RandomizedSearchCV created.


In [23]:
# ==========================================
# START HYPERPARAMETER TUNING
# ==========================================

random_search.fit(
    X_train,
    y_train
)

print("\nHyperparameter tuning completed.")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END model__colsample_bytree=0.7, model__gamma=0.3, model__learning_rate=0.08, model__max_depth=10, model__min_child_weight=3, model__n_estimators=100, model__reg_alpha=0.1, model__reg_lambda=2, model__subsample=0.9; total time=   0.4s
[CV] END model__colsample_bytree=0.7, model__gamma=0.3, model__learning_rate=0.08, model__max_depth=10, model__min_child_weight=3, model__n_estimators=100, model__reg_alpha=0.1, model__reg_lambda=2, model__subsample=0.9; total time=   0.5s
[CV] END model__colsample_bytree=0.7, model__gamma=0.3, model__learning_rate=0.08, model__max_depth=10, model__min_child_weight=3, model__n_estimators=100, model__reg_alpha=0.1, model__reg_lambda=2, model__subsample=0.9; total time=   0.4s
[CV] END model__colsample_bytree=0.7, model__gamma=0.3, model__learning_rate=0.08, model__max_depth=10, model__min_child_weight=3, model__n_estimators=100, model__reg_alpha=0.1, model__reg_lambda=2, model__subsample=0.9

In [24]:
# ==========================================
# BEST CV RESULT
# ==========================================

print(
    "Best 5-Fold CV Accuracy:",
    f"{random_search.best_score_:.6f}"
)

print("\nBest Parameters:")

for parameter, value in (
    random_search.best_params_.items()
):

    print(
        f"{parameter}: {value}"
    )

Best 5-Fold CV Accuracy: 0.858737

Best Parameters:
model__subsample: 0.9
model__reg_lambda: 2
model__reg_alpha: 0.1
model__n_estimators: 100
model__min_child_weight: 3
model__max_depth: 10
model__learning_rate: 0.08
model__gamma: 0.3
model__colsample_bytree: 0.7


In [25]:
# ==========================================
# CV RESULTS
# ==========================================

cv_results = pd.DataFrame(
    random_search.cv_results_
)

cv_summary = cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",

        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__learning_rate",
        "param_model__subsample",
        "param_model__colsample_bytree",
        "param_model__min_child_weight",
        "param_model__gamma",
        "param_model__reg_alpha",
        "param_model__reg_lambda"
    ]
].sort_values(
    "rank_test_score"
)

display(cv_summary)

,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__learning_rate,param_model__subsample,param_model__colsample_bytree,param_model__min_child_weight,param_model__gamma,param_model__reg_alpha,param_model__reg_lambda
0,1,0.858737,0.001781,100,10,0.08,0.9,0.7,3,0.3,0.10,2
9,2,0.857875,0.001321,100,10,0.15,0.8,0.7,3,0.3,1.00,5
3,3,0.857837,0.001234,200,6,0.15,0.9,0.8,10,0.5,0.00,10
1,4,0.857088,0.000933,200,5,0.08,1.0,1.0,3,0.3,1.00,1
5,5,0.857050,0.002613,100,10,0.15,1.0,1.0,3,0.1,0.00,2
4,6,0.856513,0.001722,700,8,0.08,0.9,0.8,3,0.1,0.10,5
7,7,0.856200,0.001918,100,5,0.15,0.7,1.0,1,0.5,0.01,2
6,8,0.855537,0.001440,300,5,0.05,0.7,0.7,1,0.3,1.00,10
2,9,0.855487,0.001389,200,4,0.15,0.7,0.7,1,0.3,0.01,10
8,10,0.846750,0.002523,300,3,0.03,0.9,0.7,10,0.1,0.10,2


In [26]:
cv_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgb_5fold_cv_results.csv"
    ),
    index=False
)

In [27]:
# ==========================================
# FINAL BASELINE MODEL
# ==========================================

baseline_model = (
    random_search.best_estimator_
)

print(
    "Final baseline model selected."
)

Final baseline model selected.


In [28]:
joblib.dump(
    baseline_model,

    os.path.join(
        OUTPUT_DIR,
        "xgb_baseline_100k.pkl"
    )
)

print("Baseline model saved.")

Baseline model saved.


In [29]:
# ==========================================
# BASELINE PREDICTIONS
# ==========================================

baseline_pred = (
    baseline_model.predict(X_test)
)

baseline_prob = (
    baseline_model.predict_proba(
        X_test
    )[:, 1]
)

print("Predictions generated.")

Predictions generated.


In [30]:
# ==========================================
# BASELINE PERFORMANCE
# ==========================================

baseline_performance = {

    "Accuracy":
        accuracy_score(
            y_test,
            baseline_pred
        ),

    "Precision":
        precision_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),

    "Recall":
        recall_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),

    "F1":
        f1_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),

    "ROC_AUC":
        roc_auc_score(
            y_test,
            baseline_prob
        )
}

baseline_performance_df = pd.DataFrame(
    [baseline_performance]
)

display(
    baseline_performance_df
)

,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.8603,0.866484,0.966969,0.913973,0.856217


In [31]:
print("Actual target distribution:")
print(
    y_test.value_counts()
)

print("\nPredicted distribution:")
print(
    pd.Series(
        baseline_pred
    ).value_counts()
)

print("\nPredicted distribution (%):")
print(
    pd.Series(
        baseline_pred
    ).value_counts(
        normalize=True
    )
)

Actual target distribution:
target
1    15349
0     4651
Name: count, dtype: int64

Predicted distribution:
1    17129
0     2871
Name: count, dtype: int64

Predicted distribution (%):
1    0.85645
0    0.14355
Name: proportion, dtype: float64


In [32]:
# ==========================================
# FAIRNESS ANALYZER
# ==========================================

class FairnessAnalyzer:

    def __init__(
        self,
        protected_attributes
    ):

        self.protected_attributes = (
            protected_attributes
        )

        self.fairness_metrics = {

            "Selection Rate":
                selection_rate,

            "TPR":
                self.true_positive_rate,

            "FPR":
                self.false_positive_rate
        }

    # ======================================
    # TRUE POSITIVE RATE
    # ======================================

    @staticmethod
    def true_positive_rate(
        y_true,
        y_pred
    ):

        if np.sum(
            np.asarray(y_true) == 1
        ) == 0:

            return np.nan

        return recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

    # ======================================
    # FALSE POSITIVE RATE
    # ======================================

    @staticmethod
    def false_positive_rate(
        y_true,
        y_pred
    ):

        tn, fp, fn, tp = (
            confusion_matrix(
                y_true,
                y_pred,
                labels=[0, 1]
            ).ravel()
        )

        denominator = fp + tn

        if denominator == 0:
            return np.nan

        return fp / denominator

    # ======================================
    # CORE FAIRNESS CALCULATION
    # ======================================

    def _calculate_fairness(
        self,
        y_true,
        y_pred,
        sensitive_features
    ):

        metric_frame = MetricFrame(

            metrics=self.fairness_metrics,

            y_true=y_true,

            y_pred=y_pred,

            sensitive_features=sensitive_features
        )

        group_counts = (
            pd.Series(
                sensitive_features
            )
            .value_counts()
            .sort_index()
        )

        dp_difference = (
            demographic_parity_difference(

                y_true,

                y_pred,

                sensitive_features=
                    sensitive_features
            )
        )

        dp_ratio = (
            demographic_parity_ratio(

                y_true,

                y_pred,

                sensitive_features=
                    sensitive_features
            )
        )

        eo_difference = (
            equalized_odds_difference(

                y_true,

                y_pred,

                sensitive_features=
                    sensitive_features
            )
        )

        return {

            "group_metrics":
                metric_frame.by_group,

            "group_counts":
                group_counts,

            "dp_difference":
                dp_difference,

            "dp_ratio":
                dp_ratio,

            "eo_difference":
                eo_difference
        }

    # ======================================
    # 1-WAY ANALYSIS
    # ======================================

    def one_way_analysis(
        self,
        y_true,
        y_pred,
        protected_test
    ):

        results = {}

        for attribute in (
            self.protected_attributes
        ):

            sensitive_features = (
                protected_test[
                    attribute
                ]
            )

            results[attribute] = (
                self._calculate_fairness(

                    y_true=y_true,

                    y_pred=y_pred,

                    sensitive_features=
                        sensitive_features
                )
            )

        return results

    # ======================================
    # 2-WAY ANALYSIS
    # ======================================

    def two_way_analysis(
        self,
        y_true,
        y_pred,
        protected_test
    ):

        results = {}

        combinations_2way = combinations(
            self.protected_attributes,
            2
        )

        for attr1, attr2 in (
            combinations_2way
        ):

            combination_name = (
                f"{attr1} × {attr2}"
            )

            sensitive_features = (

                protected_test[
                    attr1
                ].astype(str)

                + " × "

                + protected_test[
                    attr2
                ].astype(str)
            )

            results[
                combination_name
            ] = (
                self._calculate_fairness(

                    y_true=y_true,

                    y_pred=y_pred,

                    sensitive_features=
                        sensitive_features
                )
            )

        return results

    # ======================================
    # PRINT 1-WAY RESULTS
    # ======================================

    def print_one_way_results(
        self,
        results
    ):

        for attribute, result in (
            results.items()
        ):

            print(
                "\n" + "=" * 90
            )

            print(
                f"1-WAY FAIRNESS: "
                f"{attribute}"
            )

            print(
                "=" * 90
            )

            print(
                "\nGroup Metrics:"
            )

            display(
                result[
                    "group_metrics"
                ]
            )

            print(
                "\nGroup Counts:"
            )

            display(
                result[
                    "group_counts"
                ].to_frame("Count")
            )

            print(
                "\nFairness Metrics:"
            )

            print(
                f"DP Difference : "
                f"{result['dp_difference']:.4f}"
            )

            print(
                f"DP Ratio      : "
                f"{result['dp_ratio']:.4f}"
            )

            print(
                f"EO Difference : "
                f"{result['eo_difference']:.4f}"
            )

    # ======================================
    # PRINT 2-WAY RESULTS
    # ======================================

    def print_two_way_results(
        self,
        results
    ):

        for combination, result in (
            results.items()
        ):

            print(
                "\n" + "=" * 90
            )

            print(
                f"2-WAY FAIRNESS: "
                f"{combination}"
            )

            print(
                "=" * 90
            )

            print(
                "\nGroup Metrics:"
            )

            display(
                result[
                    "group_metrics"
                ]
            )

            print(
                "\nGroup Counts:"
            )

            display(
                result[
                    "group_counts"
                ].to_frame("Count")
            )

            print(
                "\nFairness Metrics:"
            )

            print(
                f"DP Difference : "
                f"{result['dp_difference']:.4f}"
            )

            print(
                f"DP Ratio      : "
                f"{result['dp_ratio']:.4f}"
            )

            print(
                f"EO Difference : "
                f"{result['eo_difference']:.4f}"
            )

    # ======================================
    # SUMMARY TABLE
    # ======================================

    @staticmethod
    def summary_table(
        results
    ):

        rows = []

        for attribute, result in (
            results.items()
        ):

            rows.append({

                "Attribute":
                    attribute,

                "DP Difference":
                    result[
                        "dp_difference"
                    ],

                "DP Ratio":
                    result[
                        "dp_ratio"
                    ],

                "EO Difference":
                    result[
                        "eo_difference"
                    ]
            })

        return pd.DataFrame(rows)

In [33]:
# ==========================================
# CREATE FAIRNESS ANALYZER
# ==========================================

fairness_analyzer = FairnessAnalyzer(
    protected_attributes=[
        "derived_ethnicity",
        "derived_race",
        "derived_sex",
        "applicant_age"
    ]
)

In [34]:
# ==========================================
# 1-WAY FAIRNESS
# ==========================================

baseline_one_way = (
    fairness_analyzer.one_way_analysis(

        y_true=y_test,

        y_pred=baseline_pred,

        protected_test=protected_test
    )
)

fairness_analyzer.print_one_way_results(
    baseline_one_way
)


1-WAY FAIRNESS: derived_ethnicity

Group Metrics:


,Selection Rate,TPR,FPR
derived_ethnicity,,,
Ethnicity Not Available,0.843498,0.965981,0.516878
Free Form Text Only,0.625000,1.000000,0.250000
Hispanic or Latino,0.805600,0.961361,0.453003
Joint,0.897345,0.975501,0.594828
Not Hispanic or Latino,0.864902,0.967558,0.493771



Group Counts:


,Count
derived_ethnicity,
Ethnicity Not Available,1738
Free Form Text Only,8
Hispanic or Latino,2500
Joint,565
Not Hispanic or Latino,15189



Fairness Metrics:
DP Difference : 0.2723
DP Ratio      : 0.6965
EO Difference : 0.3448

1-WAY FAIRNESS: derived_race

Group Metrics:


...


Fairness Metrics:
DP Difference : 0.0876
DP Ratio      : 0.9025
EO Difference : 0.1049

1-WAY FAIRNESS: applicant_age

Group Metrics:


,Selection Rate,TPR,FPR
applicant_age,,,
25-34,0.883908,0.975176,0.463277
35-44,0.866835,0.974919,0.486230
45-54,0.858384,0.968293,0.518732
55-64,0.841753,0.963960,0.503440
65-74,0.821839,0.947197,0.474729
<25,0.868148,0.971171,0.391667
>74,0.800208,0.928571,0.521452



Group Counts:


,Count
applicant_age,
25-34,3971
35-44,4761
45-54,4258
55-64,3286
65-74,2088
<25,675
>74,961



Fairness Metrics:
DP Difference : 0.0837
DP Ratio      : 0.9053
EO Difference : 0.1298


In [35]:
# ==========================================
# 1-WAY SUMMARY
# ==========================================

baseline_one_way_summary = (
    fairness_analyzer.summary_table(
        baseline_one_way
    )
)

display(
    baseline_one_way_summary
)

,Attribute,DP Difference,DP Ratio,EO Difference
0,derived_ethnicity,0.272345,0.696499,0.344828
1,derived_race,0.274510,0.725490,0.705882
2,derived_sex,0.087613,0.902514,0.104869
3,applicant_age,0.083700,0.905307,0.129785


In [36]:
baseline_one_way_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_one_way_summary_100k.csv"
    ),
    index=False
)

In [37]:
# ==========================================
# 2-WAY FAIRNESS
# ==========================================

baseline_two_way = (
    fairness_analyzer.two_way_analysis(

        y_true=y_test,

        y_pred=baseline_pred,

        protected_test=protected_test
    )
)

fairness_analyzer.print_two_way_results(
    baseline_two_way
)


2-WAY FAIRNESS: derived_ethnicity × derived_race

Group Metrics:


,Selection Rate,TPR,FPR
sensitive_feature_0,,,
Ethnicity Not Available × 2 or more minority races,0.400000,1.000000,0.000000
Ethnicity Not Available × American Indian or Alaska Native,0.666667,1.000000,0.444444
Ethnicity Not Available × Asian,0.810345,0.954545,0.357143
Ethnicity Not Available × Black or African American,0.612500,0.911765,0.391304
Ethnicity Not Available × Joint,0.615385,0.888889,0.000000
Ethnicity Not Available × Native Hawaiian or Other Pacific Islander,1.000000,1.000000,1.000000
Ethnicity Not Available × Race Not Available,0.876076,0.968182,0.588652
Ethnicity Not Available × White,0.818408,0.968641,0.443478
Free Form Text Only × American Indian or Alaska Native,1.000000,1.000000,NaN



Group Counts:


,Count
Ethnicity Not Available × 2 or more minority races,5
Ethnicity Not Available × American Indian or Alaska Native,15
Ethnicity Not Available × Asian,58
Ethnicity Not Available × Black or African American,80
Ethnicity Not Available × Joint,13
Ethnicity Not Available × Native Hawaiian or Other Pacific Islander,3
Ethnicity Not Available × Race Not Available,1162
Ethnicity Not Available × White,402
Free Form Text Only × American Indian or Alaska Native,1
Free Form Text Only × Asian,2



Fairness Metrics:
DP Difference : 0.7500
DP Ratio      : 0.2500
EO Difference : 1.0000

2-WAY FAIRNESS: derived_ethnicity × derived_sex

Group Metrics:


...


Fairness Metrics:
DP Difference : 1.0000
DP Ratio      : 0.0000
EO Difference : 1.0000

2-WAY FAIRNESS: derived_sex × applicant_age

Group Metrics:


,Selection Rate,TPR,FPR
sensitive_feature_0,,,
Female × 25-34,0.823674,0.963993,0.395000
Female × 35-44,0.789055,0.958571,0.400000
Female × 45-54,0.835893,0.966755,0.496552
Female × 55-64,0.823597,0.961353,0.484127
Female × 65-74,0.795262,0.948478,0.396341
Female × <25,0.809524,0.978947,0.290323
Female × >74,0.761438,0.900498,0.495238
Joint × 25-34,0.922030,0.980142,0.524272
Joint × 35-44,0.908795,0.974984,0.544170



Group Counts:


,Count
Female × 25-34,811
Female × 35-44,1005
Female × 45-54,1042
Female × 55-64,873
Female × 65-74,591
Female × <25,126
Female × >74,306
Joint × 25-34,1616
Joint × 35-44,1842
Joint × 45-54,1548



Fairness Metrics:
DP Difference : 0.1932
DP Ratio      : 0.7905
EO Difference : 0.3201


In [38]:
# ==========================================
# 2-WAY SUMMARY
# ==========================================

baseline_two_way_summary = (
    fairness_analyzer.summary_table(
        baseline_two_way
    )
)

display(
    baseline_two_way_summary
)

,Attribute,DP Difference,DP Ratio,EO Difference
0,derived_ethnicity × derived_race,0.750000,0.250000,1.000000
1,derived_ethnicity × derived_sex,0.907540,0.000000,1.000000
2,derived_ethnicity × applicant_age,1.000000,0.000000,1.000000
3,derived_race × derived_sex,0.375000,0.625000,1.000000
4,derived_race × applicant_age,1.000000,0.000000,1.000000
5,derived_sex × applicant_age,0.193156,0.790509,0.320067


In [39]:
baseline_two_way_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_two_way_summary_100k.csv"
    ),
    index=False
)

In [40]:
# ==========================================
# SAVE FAIRNESS RESULTS
# ==========================================

joblib.dump(
    baseline_one_way,

    os.path.join(
        OUTPUT_DIR,
        "baseline_one_way_results_100k.pkl"
    )
)

joblib.dump(
    baseline_two_way,

    os.path.join(
        OUTPUT_DIR,
        "baseline_two_way_results_100k.pkl"
    )
)

print("Fairness results saved.")

Fairness results saved.


In [41]:
# ==========================================
# FINAL BASELINE SUMMARY
# ==========================================

print("=" * 80)
print("100K HMDA XGBOOST BASELINE COMPLETE")
print("=" * 80)

print("\nDataset:")
print("Rows:", len(df))

print("\nCV:")
print("5-Fold Stratified CV")

print("\nModel:")
print("XGBoost")

print("\nProtected attributes excluded from training:")
print(protected_attributes)

print("\nBest CV Accuracy:")
print(
    f"{random_search.best_score_:.4f}"
)

print("\nBest Parameters:")
for parameter, value in (
    random_search.best_params_.items()
):
    print(
        f"{parameter}: {value}"
    )

print("\nTest Performance:")
display(
    baseline_performance_df
)

print("\n1-Way Fairness:")
display(
    baseline_one_way_summary
)

print("\n2-Way Fairness:")
display(
    baseline_two_way_summary
)

100K HMDA XGBOOST BASELINE COMPLETE

Dataset:
Rows: 100000

CV:
5-Fold Stratified CV

Model:
XGBoost

Protected attributes excluded from training:
['derived_ethnicity', 'derived_race', 'derived_sex', 'applicant_age']

Best CV Accuracy:
0.8587

Best Parameters:
model__subsample: 0.9
model__reg_lambda: 2
model__reg_alpha: 0.1
model__n_estimators: 100
model__min_child_weight: 3
model__max_depth: 10
model__learning_rate: 0.08
model__gamma: 0.3
model__colsample_bytree: 0.7

Test Performance:


,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.8603,0.866484,0.966969,0.913973,0.856217



1-Way Fairness:


,Attribute,DP Difference,DP Ratio,EO Difference
0,derived_ethnicity,0.272345,0.696499,0.344828
1,derived_race,0.274510,0.725490,0.705882
2,derived_sex,0.087613,0.902514,0.104869
3,applicant_age,0.083700,0.905307,0.129785



2-Way Fairness:


,Attribute,DP Difference,DP Ratio,EO Difference
0,derived_ethnicity × derived_race,0.750000,0.250000,1.000000
1,derived_ethnicity × derived_sex,0.907540,0.000000,1.000000
2,derived_ethnicity × applicant_age,1.000000,0.000000,1.000000
3,derived_race × derived_sex,0.375000,0.625000,1.000000
4,derived_race × applicant_age,1.000000,0.000000,1.000000
5,derived_sex × applicant_age,0.193156,0.790509,0.320067


BIAS MITIGATION TECHNIQUES


In [42]:
# ==========================================
# EG MITIGATION - IMPORTS
# ==========================================

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from fairlearn.reductions import (
    ExponentiatedGradient,
    DemographicParity,
    EqualizedOdds
)

from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equal_opportunity_difference
)

import numpy as np
import pandas as pd
import os

In [43]:
# ==========================================
# FIXED TUNED XGBOOST PARAMETERS
# ==========================================

BEST_PARAMS = {
    "subsample": 0.9,
    "reg_lambda": 2,
    "reg_alpha": 0.1,
    "n_estimators": 100,
    "min_child_weight": 3,
    "max_depth": 10,
    "learning_rate": 0.08,
    "gamma": 0.3,
    "colsample_bytree": 0.7
}

EPSILONS = [0.01, 0.03, 0.05, 0.10, 0.20]

print("Using fixed tuned XGBoost parameters:")
print(BEST_PARAMS)

Using fixed tuned XGBoost parameters:
{'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 10, 'learning_rate': 0.08, 'gamma': 0.3, 'colsample_bytree': 0.7}


In [44]:
# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("X_fit :", X_fit.shape)
print("X_val :", X_val.shape)
print("X_test:", X_test.shape)

X_fit : (64000, 10)
X_val : (16000, 10)
X_test: (20000, 10)


In [45]:
# ==========================================
# SENSITIVE FEATURES
# ==========================================

race = df["derived_race"]
ethnicity = df["derived_ethnicity"]

race_fit = race.loc[X_fit.index]
race_val = race.loc[X_val.index]
race_test = race.loc[X_test.index]

ethnicity_fit = ethnicity.loc[X_fit.index]
ethnicity_val = ethnicity.loc[X_val.index]
ethnicity_test = ethnicity.loc[X_test.index]

In [46]:
# ==========================================
# RACE × ETHNICITY INTERSECTION
# ==========================================

race_ethnicity = (
    df["derived_race"].astype(str)
    + " × "
    + df["derived_ethnicity"].astype(str)
)

race_ethnicity_fit = race_ethnicity.loc[X_fit.index]
race_ethnicity_val = race_ethnicity.loc[X_val.index]
race_ethnicity_test = race_ethnicity.loc[X_test.index]

print("Intersection groups:")
print(race_ethnicity_fit.value_counts())

Intersection groups:
White × Not Hispanic or Latino                                         37308
White × Hispanic or Latino                                              5789
Black or African American × Not Hispanic or Latino                      5134
Asian × Not Hispanic or Latino                                          3774
Race Not Available × Ethnicity Not Available                            3553
Race Not Available × Hispanic or Latino                                 1496
White × Joint                                                           1254
White × Ethnicity Not Available                                         1234
Joint × Not Hispanic or Latino                                          1216
Race Not Available × Not Hispanic or Latino                             1004
Black or African American × Ethnicity Not Available                      282
American Indian or Alaska Native × Not Hispanic or Latino                274
Black or African American × Hispanic or Latino         

In [47]:
# ==========================================
# FIXED XGBOOST BASE ESTIMATOR
# ==========================================

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

def create_base_xgb():

    xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,

        subsample=0.9,
        reg_lambda=2,
        reg_alpha=0.1,
        n_estimators=100,
        min_child_weight=3,
        max_depth=10,
        learning_rate=0.08,
        gamma=0.3,
        colsample_bytree=0.7
    )

    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", xgb)
    ])

In [52]:
# ==========================================
# EG TRAINING FUNCTION - CORRECTED
# ==========================================

def train_eg(
    X_fit,
    y_fit,
    sensitive_fit,
    constraint_name,
    epsilon
):

    # --------------------------------------
    # Fairness constraint
    # --------------------------------------

    if constraint_name == "DemographicParity":

        constraint = DemographicParity(
            difference_bound=epsilon
        )

    elif constraint_name == "EqualizedOdds":

        constraint = EqualizedOdds(
            difference_bound=epsilon
        )

    else:
        raise ValueError(
            "Unknown constraint"
        )

    # --------------------------------------
    # Fixed XGBoost estimator
    # --------------------------------------

    estimator = create_base_xgb()

    # --------------------------------------
    # Exponentiated Gradient
    # --------------------------------------

    eg_model = ExponentiatedGradient(
        estimator=estimator,
        constraints=constraint,

        # Pass sample weights through pipeline
        sample_weight_name="model__sample_weight",

        # EG optimization parameter
        eps=0.01,

        max_iter=50
    )

    # --------------------------------------
    # Fit
    # --------------------------------------

    eg_model.fit(
        X_fit,
        y_fit,
        sensitive_features=sensitive_fit
    )

    return eg_model

In [53]:
# ==========================================
# METRICS FUNCTION
# ==========================================

def evaluate_fairness_model(
    model,
    X,
    y,
    sensitive_features
):

    y_pred = model.predict(X)

    # EG does not expose predict_proba()
    roc_auc = np.nan

    # --------------------------------------
    # Performance
    # --------------------------------------

    accuracy = accuracy_score(y, y_pred)

    precision = precision_score(
        y,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y,
        y_pred,
        zero_division=0
    )

    # --------------------------------------
    # Confusion matrix
    # --------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y,
        y_pred,
        labels=[0, 1]
    ).ravel()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    fnr = fn / (tp + fn) if (tp + fn) > 0 else np.nan

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    selection_rate = np.mean(y_pred)

    # --------------------------------------
    # Fairness
    # --------------------------------------

    dp_difference = demographic_parity_difference(
        y,
        y_pred,
        sensitive_features=sensitive_features
    )

    dp_ratio = demographic_parity_ratio(
        y,
        y_pred,
        sensitive_features=sensitive_features
    )

    eopp_difference = equal_opportunity_difference(
        y,
        y_pred,
        sensitive_features=sensitive_features
    )

    eo_difference = equalized_odds_difference(
        y,
        y_pred,
        sensitive_features=sensitive_features
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall_TPR": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,

        "Selection_Rate": selection_rate,

        "DPR": dp_ratio,
        "Statistical_Parity_Difference": dp_difference,
        "DP_Difference": dp_difference,

        "TPR": tpr,
        "FNR": fnr,
        "FPR": fpr,

        "Equal_Opportunity_Difference": eopp_difference,
        "Equalized_Odds_Difference": eo_difference
    }

In [54]:
# ==========================================
# MITIGATION CONFIGURATIONS
# ==========================================

sensitive_configs = {

    "Race": {
        "fit": race_fit,
        "val": race_val,
        "test": race_test
    },

    "Ethnicity": {
        "fit": ethnicity_fit,
        "val": ethnicity_val,
        "test": ethnicity_test
    },

    "Race × Ethnicity": {
        "fit": race_ethnicity_fit,
        "val": race_ethnicity_val,
        "test": race_ethnicity_test
    }
}

In [55]:
# ==========================================
# RUN ALL EG VALIDATION EXPERIMENTS
# ==========================================

validation_results = []
validation_models = {}

for attribute_name, sensitive_data in sensitive_configs.items():

    for constraint_name in [
        "DemographicParity",
        "EqualizedOdds"
    ]:

        for epsilon in EPSILONS:

            print(
                f"\n{'='*70}\n"
                f"{attribute_name} | "
                f"{constraint_name} | "
                f"epsilon={epsilon}\n"
                f"{'='*70}"
            )

            try:

                model = train_eg(
                    X_fit=X_fit,
                    y_fit=y_fit,
                    sensitive_fit=sensitive_data["fit"],
                    constraint_name=constraint_name,
                    epsilon=epsilon
                )

                metrics = evaluate_fairness_model(
                    model=model,
                    X=X_val,
                    y=y_val,
                    sensitive_features=sensitive_data["val"]
                )

                validation_results.append({
                    "Attribute": attribute_name,
                    "Constraint": constraint_name,
                    "Epsilon": epsilon,
                    **metrics
                })

                validation_models[
                    (
                        attribute_name,
                        constraint_name,
                        epsilon
                    )
                ] = model

                print(
                    f"Accuracy = {metrics['Accuracy']:.4f}"
                )

                print(
                    f"DP Diff = {metrics['DP_Difference']:.4f}"
                )

                print(
                    f"DPR = {metrics['DPR']:.4f}"
                )

                print(
                    f"EO Diff = "
                    f"{metrics['Equalized_Odds_Difference']:.4f}"
                )

            except Exception as e:

                print(
                    f"ERROR: {attribute_name} | "
                    f"{constraint_name} | "
                    f"{epsilon}"
                )

                print(e)


Race | DemographicParity | epsilon=0.01
Accuracy = 0.8524
DP Diff = 0.2553
DPR = 0.7447
EO Diff = 0.9861

Race | DemographicParity | epsilon=0.03
Accuracy = 0.8550
DP Diff = 0.2619
DPR = 0.7381
EO Diff = 0.9861

Race | DemographicParity | epsilon=0.05
Accuracy = 0.8578
DP Diff = 0.2553
DPR = 0.7447
EO Diff = 0.9861

Race | DemographicParity | epsilon=0.1
Accuracy = 0.8601
DP Diff = 0.2553
DPR = 0.7447
EO Diff = 0.9852

Race | DemographicParity | epsilon=0.2
Accuracy = 0.8580
DP Diff = 0.2650
DPR = 0.7350
EO Diff = 0.9791

Race | EqualizedOdds | epsilon=0.01
Accuracy = 0.8588
DP Diff = 0.2619
DPR = 0.7381
EO Diff = 0.9861

Race | EqualizedOdds | epsilon=0.03
Accuracy = 0.8588
DP Diff = 0.2553
DPR = 0.7447
EO Diff = 0.9861

Race | EqualizedOdds | epsilon=0.05
Accuracy = 0.8588
DP Diff = 0.2553
DPR = 0.7447
EO Diff = 0.9861

Race | EqualizedOdds | epsilon=0.1
Accuracy = 0.8608
DP Diff = 0.2619
DPR = 0.7381
EO Diff = 0.9815

Race | EqualizedOdds | epsilon=0.2
Accuracy = 0.8585
DP Diff = 0

In [56]:
# ==========================================
# VALIDATION RESULTS
# ==========================================

validation_df = pd.DataFrame(
    validation_results
)

display(
    validation_df.round(4)
)

,Attribute,Constraint,Epsilon,Accuracy,Precision,Recall_TPR,F1,ROC_AUC,Selection_Rate,DPR,Statistical_Parity_Difference,DP_Difference,TPR,FNR,FPR,Equal_Opportunity_Difference,Equalized_Odds_Difference
0,Race,DemographicParity,0.01,0.8524,0.8583,0.9673,0.9096,NaN,0.8649,0.7447,0.2553,0.2553,0.9673,0.0327,0.5270,0.9861,0.9861
1,Race,DemographicParity,0.03,0.8550,0.8608,0.9676,0.9110,NaN,0.8627,0.7381,0.2619,0.2619,0.9676,0.0324,0.5165,0.9861,0.9861
2,Race,DemographicParity,0.05,0.8578,0.8639,0.9669,0.9125,NaN,0.8589,0.7447,0.2553,0.2553,0.9669,0.0331,0.5026,0.9861,0.9861
3,Race,DemographicParity,0.10,0.8601,0.8662,0.9670,0.9138,NaN,0.8568,0.7447,0.2553,0.2553,0.9670,0.0330,0.4929,0.9852,0.9852
4,Race,DemographicParity,0.20,0.8580,0.8697,0.9586,0.9120,NaN,0.8459,0.7350,0.2650,0.2650,0.9586,0.0414,0.4741,0.9791,0.9791
5,Race,EqualizedOdds,0.01,0.8588,0.8665,0.9645,0.9129,NaN,0.8542,0.7381,0.2619,0.2619,0.9645,0.0355,0.4902,0.9861,0.9861
6,Race,EqualizedOdds,0.03,0.8588,0.8665,0.9646,0.9129,NaN,0.8543,0.7447,0.2553,0.2553,0.9646,0.0354,0.4905,0.9861,0.9861
7,Race,EqualizedOdds,0.05,0.8588,0.8660,0.9654,0.9130,NaN,0.8556,0.7447,0.2553,0.2553,0.9654,0.0346,0.4931,0.9861,0.9861
8,Race,EqualizedOdds,0.10,0.8608,0.8680,0.9653,0.9141,NaN,0.8534,0.7381,0.2619,0.2619,0.9653,0.0347,0.4843,0.9815,0.9815
9,Race,EqualizedOdds,0.20,0.8585,0.8666,0.9640,0.9127,NaN,0.8537,0.7436,0.2564,0.2564,0.9640,0.0360,0.4897,0.9852,0.9852


In [57]:
# ==========================================
# BASELINE VALIDATION
# ==========================================

baseline_val_pred = baseline_model.predict(
    X_val
)

baseline_val_results = {}

for attribute_name, sensitive_data in sensitive_configs.items():

    baseline_val_results[
        attribute_name
    ] = evaluate_fairness_model(
        model=baseline_model,
        X=X_val,
        y=y_val,
        sensitive_features=sensitive_data["val"]
    )

display(
    pd.DataFrame(baseline_val_results).T.round(4)
)

,Accuracy,Precision,Recall_TPR,F1,ROC_AUC,Selection_Rate,DPR,Statistical_Parity_Difference,DP_Difference,TPR,FNR,FPR,Equal_Opportunity_Difference,Equalized_Odds_Difference
Race,0.8798,0.8792,0.9777,0.9258,NaN,0.8534,0.7350,0.2650,0.2650,0.9777,0.0223,0.4432,1.0000,1.0000
Ethnicity,0.8798,0.8792,0.9777,0.9258,NaN,0.8534,0.7967,0.2033,0.2033,0.9777,0.0223,0.4432,0.0313,0.4553
Race × Ethnicity,0.8798,0.8792,0.9777,0.9258,NaN,0.8534,0.3333,0.6667,0.6667,0.9777,0.0223,0.4432,1.0000,1.0000


In [58]:
# ==========================================
# SELECT EPSILON
# ==========================================

ACCURACY_TOLERANCE = 0.02

selected_configs = {}

for attribute_name in sensitive_configs:

    baseline_accuracy = (
        baseline_val_results[
            attribute_name
        ]["Accuracy"]
    )

    minimum_accuracy = (
        baseline_accuracy
        - ACCURACY_TOLERANCE
    )

    for constraint_name in [
        "DemographicParity",
        "EqualizedOdds"
    ]:

        candidates = validation_df[
            (validation_df["Attribute"] == attribute_name) &
            (validation_df["Constraint"] == constraint_name) &
            (validation_df["Accuracy"] >= minimum_accuracy)
        ].copy()

        if len(candidates) == 0:

            print(
                f"No model within accuracy tolerance for "
                f"{attribute_name} + {constraint_name}"
            )

            continue

        if constraint_name == "DemographicParity":

            candidates = candidates.sort_values(
                ["DP_Difference", "Accuracy"],
                ascending=[True, False]
            )

        else:

            candidates = candidates.sort_values(
                ["Equalized_Odds_Difference", "Accuracy"],
                ascending=[True, False]
            )

        selected = candidates.iloc[0]

        selected_configs[
            (
                attribute_name,
                constraint_name
            )
        ] = selected["Epsilon"]


print("\nSELECTED EPSILONS")
print("=" * 60)

for key, epsilon in selected_configs.items():

    print(
        f"{key[0]:<25} "
        f"{key[1]:<22} "
        f"epsilon = {epsilon}"
    )

No model within accuracy tolerance for Race × Ethnicity + DemographicParity

SELECTED EPSILONS
Race                      DemographicParity      epsilon = 0.1
Race                      EqualizedOdds          epsilon = 0.1
Ethnicity                 DemographicParity      epsilon = 0.1
Ethnicity                 EqualizedOdds          epsilon = 0.05
Race × Ethnicity          EqualizedOdds          epsilon = 0.03


In [59]:
# ==========================================
# FINAL TEST EVALUATION
# ==========================================

final_eg_results = []

for (
    attribute_name,
    constraint_name
), epsilon in selected_configs.items():

    model = validation_models[
        (
            attribute_name,
            constraint_name,
            epsilon
        )
    ]

    sensitive_test = sensitive_configs[
        attribute_name
    ]["test"]

    metrics = evaluate_fairness_model(
        model=model,
        X=X_test,
        y=y_test,
        sensitive_features=sensitive_test
    )

    final_eg_results.append({
        "Model": "EG",
        "Attribute": attribute_name,
        "Constraint": constraint_name,
        "Epsilon": epsilon,
        **metrics
    })

final_eg_df = pd.DataFrame(
    final_eg_results
)

display(
    final_eg_df.round(4)
)

,Model,Attribute,Constraint,Epsilon,Accuracy,Precision,Recall_TPR,F1,ROC_AUC,Selection_Rate,DPR,Statistical_Parity_Difference,DP_Difference,TPR,FNR,FPR,Equal_Opportunity_Difference,Equalized_Odds_Difference
0,EG,Race,DemographicParity,0.10,0.8600,0.8658,0.9676,0.9139,NaN,0.8576,0.8122,0.1678,0.1678,0.9676,0.0324,0.4947,0.0566,0.4314
1,EG,Race,EqualizedOdds,0.10,0.8589,0.8653,0.9666,0.9132,NaN,0.8574,0.6667,0.3333,0.3333,0.9666,0.0334,0.4967,0.1176,0.7647
2,EG,Ethnicity,DemographicParity,0.10,0.8601,0.8661,0.9673,0.9139,NaN,0.8572,0.6910,0.2794,0.2794,0.9673,0.0327,0.4937,0.0392,0.3534
3,EG,Ethnicity,EqualizedOdds,0.05,0.8586,0.8678,0.9624,0.9127,NaN,0.8511,0.6938,0.2759,0.2759,0.9624,0.0376,0.4838,0.0415,0.3621
4,EG,Race × Ethnicity,EqualizedOdds,0.03,0.8588,0.8661,0.9652,0.9130,NaN,0.8553,0.4000,0.6000,0.6000,0.9652,0.0348,0.4926,1.0000,1.0000


In [60]:
# ==========================================
# BASELINE TEST RESULTS
# ==========================================

baseline_test_results = []

baseline_test_pred = baseline_model.predict(
    X_test
)

for attribute_name, sensitive_data in sensitive_configs.items():

    metrics = evaluate_fairness_model(
        model=baseline_model,
        X=X_test,
        y=y_test,
        sensitive_features=sensitive_data["test"]
    )

    baseline_test_results.append({
        "Model": "Baseline XGBoost",
        "Attribute": attribute_name,
        "Constraint": "None",
        "Epsilon": np.nan,
        **metrics
    })

baseline_test_df = pd.DataFrame(
    baseline_test_results
)

In [61]:
# ==========================================
# FINAL COMPARISON
# ==========================================

comparison_df = pd.concat(
    [
        baseline_test_df,
        final_eg_df
    ],
    ignore_index=True
)

comparison_df = comparison_df[
    [
        "Model",
        "Attribute",
        "Constraint",
        "Epsilon",

        "Accuracy",
        "Precision",
        "Recall_TPR",
        "F1",
        "ROC_AUC",

        "Selection_Rate",
        "DPR",
        "Statistical_Parity_Difference",
        "DP_Difference",

        "TPR",
        "FNR",
        "FPR",

        "Equal_Opportunity_Difference",
        "Equalized_Odds_Difference"
    ]
]

display(
    comparison_df.round(4)
)

,Model,Attribute,Constraint,Epsilon,Accuracy,Precision,Recall_TPR,F1,ROC_AUC,Selection_Rate,DPR,Statistical_Parity_Difference,DP_Difference,TPR,FNR,FPR,Equal_Opportunity_Difference,Equalized_Odds_Difference
0,Baseline XGBoost,Race,None,NaN,0.8603,0.8665,0.9670,0.9140,NaN,0.8564,0.7255,0.2745,0.2745,0.9670,0.0330,0.4917,0.0588,0.7059
1,Baseline XGBoost,Ethnicity,None,NaN,0.8603,0.8665,0.9670,0.9140,NaN,0.8564,0.6965,0.2723,0.2723,0.9670,0.0330,0.4917,0.0386,0.3448
2,Baseline XGBoost,Race × Ethnicity,None,NaN,0.8603,0.8665,0.9670,0.9140,NaN,0.8564,0.2500,0.7500,0.7500,0.9670,0.0330,0.4917,1.0000,1.0000
3,EG,Race,DemographicParity,0.10,0.8600,0.8658,0.9676,0.9139,NaN,0.8576,0.8122,0.1678,0.1678,0.9676,0.0324,0.4947,0.0566,0.4314
4,EG,Race,EqualizedOdds,0.10,0.8589,0.8653,0.9666,0.9132,NaN,0.8574,0.6667,0.3333,0.3333,0.9666,0.0334,0.4967,0.1176,0.7647
5,EG,Ethnicity,DemographicParity,0.10,0.8601,0.8661,0.9673,0.9139,NaN,0.8572,0.6910,0.2794,0.2794,0.9673,0.0327,0.4937,0.0392,0.3534
6,EG,Ethnicity,EqualizedOdds,0.05,0.8586,0.8678,0.9624,0.9127,NaN,0.8511,0.6938,0.2759,0.2759,0.9624,0.0376,0.4838,0.0415,0.3621
7,EG,Race × Ethnicity,EqualizedOdds,0.03,0.8588,0.8661,0.9652,0.9130,NaN,0.8553,0.4000,0.6000,0.6000,0.9652,0.0348,0.4926,1.0000,1.0000
